# Binary Search Trees

```{contents}
:local:
:depth: 2
```


## The BST Invariant

```{index} binary search tree; BST invariant
```


A **binary search tree** is a binary tree with an ordering rule, often called an **invariant**:

- every value in a node's left subtree is less than the node's value;
- every value in a node's right subtree is greater than the node's value.

This rule must be true at every node, not only at the root.


In [ ]:
using System;

var root = new Node(8);
root.Left = new Node(3);
root.Right = new Node(10);
root.Left.Left = new Node(1);
root.Left.Right = new Node(6);
root.Right.Right = new Node(14);

Console.WriteLine(IsValidBst(root, int.MinValue, int.MaxValue));

bool IsValidBst(Node? node, int min, int max)
{
    if (node is null)
    {
        return true;
    }

    if (node.Value <= min || node.Value >= max)
    {
        return false;
    }

    return IsValidBst(node.Left, min, node.Value)
        && IsValidBst(node.Right, node.Value, max);
}

class Node
{
    public int Value { get; }
    public Node? Left { get; set; }
    public Node? Right { get; set; }

    public Node(int value) => Value = value;
}


The valid range narrows as the recursion moves down the tree. A right child must be greater than its parent and also remain within any limits inherited from ancestors.


## Search

```{index} binary search tree; search
```


BST search uses comparison to discard one subtree at each step. If the target is smaller than the current node, search left. If it is larger, search right. If it is equal, the value has been found.


In [ ]:
using System;

var tree = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    tree.Insert(value);
}

Console.WriteLine(tree.Contains(7));
Console.WriteLine(tree.Contains(2));

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value)
    {
        root = Insert(root, value);
    }

    public bool Contains(int value)
    {
        Node? current = root;
        while (current is not null)
        {
            if (value == current.Value) return true;
            current = value < current.Value ? current.Left : current.Right;
        }

        return false;
    }

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }

        public Node(int value) => Value = value;
    }
}


This `Contains` method is iterative: it walks one path from the root down. A recursive version would follow the same comparisons.


## Insertion

```{index} binary search tree; insertion
```


Insertion searches for the place where the value belongs. When the search reaches an empty child reference, the new node is attached there.

This implementation ignores duplicate values. Another design could count duplicates or choose a consistent side for equal values.


In [ ]:
using System;
using System.Collections.Generic;

var tree = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    tree.Insert(value);
}

Console.WriteLine(string.Join(", ", tree.InOrder()));

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value)
    {
        root = Insert(root, value);
    }

    public List<int> InOrder()
    {
        var values = new List<int>();
        InOrder(root, values);
        return values;
    }

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private static void InOrder(Node? node, List<int> values)
    {
        if (node is null) return;
        InOrder(node.Left, values);
        values.Add(node.Value);
        InOrder(node.Right, values);
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }

        public Node(int value) => Value = value;
    }
}


For a BST, inorder traversal visits values in sorted order. That makes it a useful check when testing insertion.


## Shape and Performance

```{index} binary search tree; height; balanced tree
```


A BST is efficient when its height stays small. If inserted values arrive in sorted order, the tree can become a chain. Then search behaves like scanning a linked list.


In [ ]:
using System;

var balancedLike = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    balancedLike.Insert(value);
}

var chain = new BinarySearchTree();
foreach (int value in new[] { 1, 3, 4, 6, 7, 8, 10, 13, 14 })
{
    chain.Insert(value);
}

Console.WriteLine($"Mixed insertion height: {balancedLike.Height()}");
Console.WriteLine($"Sorted insertion height: {chain.Height()}");

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value) => root = Insert(root, value);
    public int Height() => Height(root);

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private static int Height(Node? node)
    {
        if (node is null) return -1;
        return 1 + Math.Max(Height(node.Left), Height(node.Right));
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }
        public Node(int value) => Value = value;
    }
}


Balanced tree structures, such as AVL trees and red-black trees, add rules that keep height under control. This chapter focuses on the basic BST first so the ordering idea is clear.


## Removal Preview

```{index} binary search tree; removal
```


Removing from a BST has more cases than search or insertion:

| Case | Basic idea |
|---|---|
| Leaf node | Remove the parent's reference to it. |
| One child | Replace the node with its child. |
| Two children | Replace the node's value with its inorder successor or predecessor, then remove that moved value. |

The two-child case is why removal is often taught after search and insertion. It is not conceptually impossible, but it has more structural bookkeeping.


In [ ]:
using System;

var root = new Node(8);
root.Left = new Node(3);
root.Right = new Node(10);
root.Right.Right = new Node(14);
root.Right.Right.Left = new Node(13);

Console.WriteLine($"Minimum: {Minimum(root)}");
Console.WriteLine($"Maximum: {Maximum(root)}");

int Minimum(Node node)
{
    Node current = node;
    while (current.Left is not null)
    {
        current = current.Left;
    }

    return current.Value;
}

int Maximum(Node node)
{
    Node current = node;
    while (current.Right is not null)
    {
        current = current.Right;
    }

    return current.Value;
}

class Node
{
    public int Value { get; }
    public Node? Left { get; set; }
    public Node? Right { get; set; }

    public Node(int value) => Value = value;
}


Finding the minimum and maximum helps explain removal: an inorder successor is the smallest value in the right subtree.


## Exercise

```{index} binary search tree; exercise
```


In [ ]:
// Exercise: Complete search
// Fill in Contains so it returns true for 7 and false for 2.

using System;

var tree = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    tree.Insert(value);
}

Console.WriteLine(tree.Contains(7));
Console.WriteLine(tree.Contains(2));

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value)
    {
        root = Insert(root, value);
    }

    public bool Contains(int value)
    {
        Node? current = root;
        while (current is not null)
        {
            // Your code starts here.



            // Your code ends here.
        }

        return false;
    }

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }
        public Node(int value) => Value = value;
    }
}


In [ ]:
// Solution
using System;

var tree = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    tree.Insert(value);
}

Console.WriteLine(tree.Contains(7));
Console.WriteLine(tree.Contains(2));

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value)
    {
        root = Insert(root, value);
    }

    public bool Contains(int value)
    {
        Node? current = root;
        while (current is not null)
        {
            if (value == current.Value)
            {
                return true;
            }

            current = value < current.Value ? current.Left : current.Right;
        }

        return false;
    }

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }
        public Node(int value) => Value = value;
    }
}


## Summary

A BST adds an ordering invariant to a binary tree. Search and insertion follow one path when the tree is well shaped, but sorted insertion can produce a tall chain.


```{rubric} Footnotes
```

[^1]: Self-balancing BSTs keep height small by rotating nodes after insertions or removals. They are a later data-structures topic.
